# Fine-tune Qwen2.5-VL-3B on the HCFA-1500 dataset (QLoRA)

End-to-end Colab notebook for the `catochris/hcfa-1500` dataset:

1. Load the dataset straight from the Hugging Face Hub.
2. QLoRA fine-tune **Qwen/Qwen2.5-VL-3B-Instruct** (image + prompt → flat JSON).
3. Run inference on the `test` split and score it with **your own `hcfa_eval` harness**
   (per-tier, per-field-class, CER, JSON-validity).

**Auto-adapts to the GPU.** Cell 2 detects the runtime and sets a profile
(resolution / epochs / batch / dtype). Just pick a GPU and run top to bottom:

| Runtime | Resolution | Epochs | Notes |
|---|---|---|---|
| **T4** (free, 15 GB) | 640px | 1 | fp16, conservative — finishes inside the free time limit |
| **L4** (Colab Pro, 24 GB) | 1280px | 3 | bf16, full quality — the recommended sweet spot |
| **A100** (40 GB) | 1280px | 3 | bf16, batch 2 — fastest; overkill for 3B, burns more units |

> Resolution is the main accuracy lever for these dense small-text forms, so the profile
> spends extra VRAM on pixels first. On a 24 GB GPU you can push `max_pixels_mult` higher
> still (cell 2) if you have headroom.

## 1. Install dependencies

In [ ]:
# Qwen2.5-VL needs transformers>=4.49. trl/peft/bitsandbytes for QLoRA SFT.
%pip install -q -U "transformers>=4.49" "trl>=0.12" "peft>=0.13" accelerate bitsandbytes datasets qwen-vl-utils
# hcfa_eval (+ hcfa_synth schema) so we can score with the repo's harness in-notebook:
%pip install -q "git+https://github.com/chrscato/hcfa-1500-reader.git"

## 2. Detect GPU → pick a hardware profile

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reclaim fragmented VRAM (avoids T4 OOM)
import torch

assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU."
name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()          # False on T4 (Turing); True on L4 / A100
DTYPE = torch.bfloat16 if BF16 else torch.float16

# Adapt resolution / epochs / batch to the GPU so you can switch runtimes without
# editing other cells. Resolution is the main accuracy lever, so spend VRAM on pixels.
if total_gb < 20:          # T4 ~15 GB (free) — conservative: fp16, low res, 1 epoch
    PROFILE = dict(tier="T4", max_pixels_mult=640, epochs=1, batch=1, grad_accum=8, eval="no")
elif total_gb < 34:        # L4 ~24 GB (Colab Pro) — full res, 3 epochs  <-- recommended
    PROFILE = dict(tier="L4", max_pixels_mult=1280, epochs=3, batch=1, grad_accum=8, eval="epoch")
else:                      # A100 40 GB+ — full res, bigger batch
    PROFILE = dict(tier="A100", max_pixels_mult=1280, epochs=3, batch=2, grad_accum=8, eval="epoch")

print(f"GPU: {name} ({total_gb:.0f} GB) | bf16={BF16} -> {DTYPE}")
print("profile:", PROFILE)

## 3. Authenticate + load the dataset

`login()` opens a token box — paste a token from https://huggingface.co/settings/tokens
(a **read** token is enough to load; you need **write** later to push the adapter).

In [ ]:
from huggingface_hub import login
login()  # paste your HF token

from datasets import load_dataset

REPO_ID = "catochris/hcfa-1500"
ds = load_dataset(REPO_ID)
print(ds)

In [ ]:
# Peek at one example: an image, the prompt, and the flat-JSON target.
ex = ds["train"][0]
print("columns:", ds["train"].column_names)
print("tier:", ex["tier"], "| sample_id:", ex["sample_id"])
print("image size:", ex["image"].size)
print("target (head):", ex["target"][:300], "...")
ex["image"]

## 4. Prompt mode (parity knob)

The dataset's `prompt` column lists all ~118 schema keys (great for *zero-shot* eval).
For **fine-tuning** the model learns the schema, so a short prompt is cheaper and trains
faster. `get_prompt()` is used in BOTH training and inference, so train/serve stay
identical whichever mode you pick.

In [ ]:
PROMPT_MODE = "minimal"  # "minimal" for fine-tuning; "schema" = full key list (zero-shot eval)

MINIMAL_PROMPT = (
    "Extract every field from this CMS-1500 (HCFA) claim form as a single flat JSON "
    'object. Use "" for any blank field. Return only the JSON.'
)


def get_prompt(example):
    return MINIMAL_PROMPT if PROMPT_MODE == "minimal" else example["prompt"]


print("PROMPT_MODE =", PROMPT_MODE)

## 5. Load Qwen2.5-VL-3B in 4-bit (QLoRA base)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# Resolution cap comes from the GPU profile (cell 2): T4 640 / L4-A100 1280. The forms
# render at 2550x3300; more pixels = sharper small text but more vision tokens + VRAM.
# On a 24 GB GPU with headroom you can bump PROFILE["max_pixels_mult"] toward 1600+.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = PROFILE["max_pixels_mult"] * 28 * 28

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,       # float16 on T4, bfloat16 on L4/A100
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    torch_dtype=DTYPE,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"
print("loaded", MODEL_ID, "| max_pixels =", MAX_PIXELS)

## 6. Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False  # required with gradient checkpointing

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Adapt the language tower only; leave the vision encoder frozen (cheaper, stabler).
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 7. Collator: image + prompt → masked-label batch

We build Qwen chat messages on the fly (image inline so `process_vision_info` can find
it), render them with the chat template, then mask the prompt-side and image tokens so
loss is computed only on the target JSON.

In [ ]:
from qwen_vl_utils import process_vision_info

IMAGE_PAD_ID = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")


def to_messages(example, with_target=True):
    msgs = [{
        "role": "user",
        "content": [
            {"type": "image", "image": example["image"]},
            {"type": "text", "text": get_prompt(example)},
        ],
    }]
    if with_target:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": example["target"]}]})
    return msgs


def collate_fn(examples):
    batch_msgs = [to_messages(ex) for ex in examples]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch_msgs]
    images = []
    for m in batch_msgs:
        imgs, _ = process_vision_info(m)
        images.extend(imgs)
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == IMAGE_PAD_ID] = -100  # don't train on vision placeholder tokens
    batch["labels"] = labels
    return batch


# sanity check one batch (also a quick VRAM + sequence-length canary)
_b = collate_fn([ds["train"][0], ds["train"][1]])
print({k: tuple(v.shape) for k, v in _b.items() if hasattr(v, "shape")})
print("seq len:", _b["input_ids"].shape[1], "(must be < MAX_LEN below or the target truncates)")

## 8. Train (QLoRA SFT)

Time scales with the profile: **L4 ~1.5–2.5 h** for 3 epochs; **free T4 ~1.5–2 h** for the
auto-selected 1 epoch. Adapter saves every epoch to `qwen2.5vl-3b-hcfa/`, so a disconnect
isn't fatal. CUDA OOM? Lower `PROFILE["max_pixels_mult"]` in cell 2 and re-run from cell 5.

In [ ]:
from trl import SFTConfig, SFTTrainer

MAX_LEN = 4096  # prompt + JSON target + vision tokens; ample for the minimal prompt

args = SFTConfig(
    output_dir="qwen2.5vl-3b-hcfa",
    num_train_epochs=PROFILE["epochs"],
    per_device_train_batch_size=PROFILE["batch"],
    per_device_eval_batch_size=1,       # eval upcasts logits to fp32; bs>1 OOMs on the loss spike
    gradient_accumulation_steps=PROFILE["grad_accum"],
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=BF16,                          # auto: bf16 on L4/A100, fp16 on T4
    fp16=not BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},  # required for Qwen2.5-VL + grad ckpt
    optim="paged_adamw_8bit",           # 8-bit optimizer states — saves VRAM
    logging_steps=5,
    eval_strategy=PROFILE["eval"],
    save_strategy="epoch",
    dataset_kwargs={"skip_prepare_dataset": True},  # our collator does the work
    remove_unused_columns=False,
    report_to="none",
)

# The max-sequence-length field was renamed max_seq_length -> max_length across trl
# versions. Set whichever this install exposes so the notebook is version-proof.
for _name in ("max_length", "max_seq_length"):
    if hasattr(args, _name):
        setattr(args, _name, MAX_LEN)
        break

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"] if PROFILE["eval"] != "no" else None,
    data_collator=collate_fn,
    processing_class=processor,
)
trainer.train()

## 9. Save / push the adapter

In [ ]:
ADAPTER_DIR  = "qwen2.5vl-3b-hcfa-adapter"
ADAPTER_REPO = "catochris/qwen2.5vl-3b-hcfa-lora"   # private Hub copy = your training safety net

trainer.save_model(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

# Push the LoRA adapter to the Hub NOW (needs a WRITE token; login() ran in cell 6) so a
# dropped runtime can't cost you the training. The adapter is tiny (~tens of MB).
model.push_to_hub(ADAPTER_REPO, private=True)
processor.push_to_hub(ADAPTER_REPO, private=True)
print("saved adapter ->", ADAPTER_DIR, "| pushed ->", ADAPTER_REPO)

## 10. Inference on the test split

Greedy decode, then pull the JSON object out of the model's text. We keep the raw
output too so the scorer can measure true JSON-validity.

In [ ]:
import json, re

model.config.use_cache = True
model.eval()


def extract_json(text):
    """Parse the model's JSON, salvaging truncated output (keep complete pairs)."""
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?", "", t).strip().strip("`").strip()
    start = t.find("{")
    if start == -1:
        return None
    frag = t[start:]
    try:
        return json.loads(frag)
    except Exception:
        pass
    # generation got cut off mid-object: keep all complete "key": "value" pairs, close brace
    cut = frag.rfind('",')
    if cut != -1:
        try:
            return json.loads(frag[:cut + 1] + "}")
        except Exception:
            pass
    return None


@torch.no_grad()
def predict_batch(examples, max_new_tokens=2048):
    """Greedy-generate JSON for several forms at once (batched = far faster)."""
    msgs = [to_messages(ex, with_target=False) for ex in examples]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]
    images = []
    for m in msgs:
        imgs, _ = process_vision_info(m)
        images.extend(imgs)
    processor.tokenizer.padding_side = "left"   # REQUIRED for correct batched generation
    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        stop_strings=["}"],            # flat JSON ends at one '}' — stop there, not at the cap
        tokenizer=processor.tokenizer,  # required for stop_strings
    )
    gen = out[:, inputs["input_ids"].shape[1]:]
    return [s.strip() for s in processor.batch_decode(gen, skip_special_tokens=True)]


# Sanity check ONE sample: confirm we now get ~118 parsed keys before the full run.
_raw = predict_batch([ds["test"][0]])[0]
print("parsed keys:", len(extract_json(_raw) or {}), "(want ~118)")
print(_raw[:400], "...")


## 11. Score with the repo's `hcfa_eval` harness

The HF dataset carries the flat-JSON `target`, so we rebuild a tiny `batch_dir`
(`{"logical": unflatten(target)}` per sample) + a split file, then call `summarize()`
— the same per-tier / per-class / CER / validity report the CLI prints. Each prediction
line includes `raw` so JSON-validity reflects the model's actual output.

In [ ]:
from pathlib import Path
from hcfa_eval.schema import unflatten
from hcfa_eval.scoring import summarize, format_summary, write_summary_csv

EVAL_BATCH = 8  # forms per generate() call — lower to 4 if you OOM, raise to 16 if you have headroom

batch_dir = Path("eval_batch"); batch_dir.mkdir(exist_ok=True)
test = ds["test"]
split_rows, pred_lines = [], []

# rebuild GT for the harness from the dataset target (flatten(unflatten(x)) == x for values)
for ex in test:
    logical = unflatten(json.loads(ex["target"]))
    (batch_dir / f"{ex['sample_id']}.json").write_text(json.dumps({"logical": logical}), encoding="utf-8")
    split_rows.append({"sample_id": ex["sample_id"], "tier": ex["tier"], "json": f"{ex['sample_id']}.json"})

# generate in batches
for i in range(0, len(test), EVAL_BATCH):
    chunk = [test[j] for j in range(i, min(i + EVAL_BATCH, len(test)))]
    for ex, raw in zip(chunk, predict_batch(chunk)):
        parsed = extract_json(raw)
        pred_lines.append(json.dumps({
            "sample_id": ex["sample_id"],
            "fields": parsed if isinstance(parsed, dict) else {},
            "raw": raw,
        }))
    print(f"  predicted {min(i + EVAL_BATCH, len(test))}/{len(test)}")

Path("test_split.jsonl").write_text("\n".join(json.dumps(r) for r in split_rows), encoding="utf-8")
Path("preds.jsonl").write_text("\n".join(pred_lines), encoding="utf-8")

summary = summarize(Path("test_split.jsonl"), Path("preds.jsonl"), batch_dir, model=f"qwen2.5vl-3b-{PROFILE['tier'].lower()}")
print(format_summary(summary))
write_summary_csv(summary, Path("runs.csv"))
print("\nwrote runs.csv")


## 11b. Merge LoRA → fp16 and push the deployable model

Run this **after** scoring — it deletes the 4-bit training model from VRAM. A QLoRA adapter
cannot be merged into a 4-bit base, so we reload the base in full precision on CPU, merge,
and push one self-contained model. That merged repo is the artifact the Modal worker loads.

In [ ]:
import gc, torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

MERGED_DIR  = "qwen2.5vl-3b-hcfa-merged"
MERGED_REPO = "catochris/qwen2.5vl-3b-hcfa"          # merged = what Modal loads

# Free the 4-bit training model + optimizer from VRAM (eval/scoring above is already done).
for _v in ("trainer", "model"):
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

# Merge in FULL PRECISION on CPU: you cannot correctly merge a LoRA into a 4-bit base, and a
# CPU merge of a 3B model is quick and can't OOM the GPU.
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map="cpu",
)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)

# Save the processor WITH the training pixel budget baked in, so the serving side
# preprocesses at the same resolution the model was trained on (MIN/MAX_PIXELS from cell 11).
proc = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
proc.save_pretrained(MERGED_DIR)

merged.push_to_hub(MERGED_REPO, private=True)
proc.push_to_hub(MERGED_REPO, private=True)
print("merged model ->", MERGED_DIR, "| pushed ->", MERGED_REPO)

## 12. Where to go next

- **Resolution sweep** — on a 24 GB GPU, raise `PROFILE["max_pixels_mult"]` (cell 2) toward
  1600+ and re-score; the brief flags resolution (not param count) as the main lever for
  the dense small-text fields (codes/money/NPIs).
- **Per-tier gap** — compare `pristine` vs `worst` in the summary; that's the
  degradation-robustness number that matters for production.
- **Overfitting** — only 404 synthetic examples; if `val` loss turns up, cut epochs or
  LoRA rank.
- **The real gate** — the data is 100% synthetic. Validate on a few real de-identified
  forms through this same harness before trusting any production numbers.